# Scenario Dreamer Risk Variance Study (Colab)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/achyutmorang/scenario-dreamer-decision-layer/blob/main/notebooks/scenario_dreamer_risk_study_colab.ipynb)

This notebook runs the next two near-term experiments on top of the frozen Scenario Dreamer + CtRL-Sim + IDM baseline.

Goals:
- quantify per-scene risk instability across repeated seeds,
- estimate an offline selector upper bound by choosing the safest sampled future among the first `K` rollouts,
- decide whether a stronger online decision-layer experiment is justified.

Important scope note:
- the selector probe here is **not** a full online replanning policy,
- it is an **offline upper bound** over sampled futures under the same fixed baseline stack.


In [ ]:
# Step 1: Sync this repo into the Colab runtime
import os
import subprocess
import sys

REPO_URL = 'https://github.com/achyutmorang/scenario-dreamer-decision-layer.git'
REPO_DIR = '/content/scenario-dreamer-decision-layer'

if os.path.isdir(REPO_DIR):
    subprocess.run(['git', '-C', REPO_DIR, 'fetch', 'origin'], check=True)
    subprocess.run(['git', '-C', REPO_DIR, 'checkout', 'main'], check=True)
    subprocess.run(['git', '-C', REPO_DIR, 'pull', '--ff-only', 'origin', 'main'], check=True)
else:
    subprocess.run(['git', 'clone', '--depth', '1', '-b', 'main', REPO_URL, REPO_DIR], check=True)

os.chdir(REPO_DIR)
for p in (REPO_DIR, os.path.join(REPO_DIR, 'src')):
    if p not in sys.path:
        sys.path.insert(0, p)

print('repo_rev:', subprocess.check_output(['git', '-C', REPO_DIR, 'rev-parse', '--short', 'HEAD'], text=True).strip())


In [ ]:
# Suggested defaults for the risk variance study
%env SCENARIO_DREAMER_DRIVE_ROOT=/content/drive/MyDrive/scenario_dreamer_research
%env SCENARIO_DREAMER_RUN_SETUP=1
%env SCENARIO_DREAMER_STUDY_SCENARIO_INDICES=0,1,2,3,4
%env SCENARIO_DREAMER_STUDY_SEEDS=0,1,2,3,4,5,6,7,8,9
%env SCENARIO_DREAMER_STUDY_SELECTOR_KS=1,2,5,10
%env SCENARIO_DREAMER_STUDY_RISK_KEY=min_ttc_proxy_s
%env SCENARIO_DREAMER_STUDY_VISUALIZE=0
%env SCENARIO_DREAMER_STUDY_NO_LIGHTWEIGHT=0


In [ ]:
# Step 2: Mount Drive, clone/pin upstream Scenario Dreamer, and bind Drive-backed assets/results
import json
import os
import subprocess
import sys

from google.colab import drive

drive.mount('/content/drive', force_remount=False)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-e', REPO_DIR], check=True)

from scenario_dreamer_decision_layer.colab import bind_drive_layout, inspect_bound_layout

subprocess.run([sys.executable, 'scripts/bootstrap_baseline.py', '--clone-upstream', '--write-lock'], check=True)
binding = bind_drive_layout(os.environ['SCENARIO_DREAMER_DRIVE_ROOT'])
print('binding:')
print(json.dumps(binding, indent=2, sort_keys=True))
print('inspection:')
print(json.dumps(inspect_bound_layout(), indent=2, sort_keys=True))


In [ ]:
# Step 3: Install a lean runtime for Scenario Dreamer simulation on Colab
import os
import subprocess
import sys

RUN_SETUP = os.environ.get('SCENARIO_DREAMER_RUN_SETUP', '1').strip().lower() in {'1', 'true', 'yes', 'y', 'on'}
print('RUN_SETUP:', RUN_SETUP)
if RUN_SETUP:
    subprocess.run([sys.executable, 'scripts/setup_colab_runtime.py', '--editable-project'], check=True)
else:
    print('Skipping runtime setup.')


In [ ]:
# Step 4: Confirm assets and scenario availability before the study
import json
import subprocess
import sys

raw = subprocess.check_output([sys.executable, 'scripts/run_smoke_eval.py', '--dry-run'], text=True)
print(raw)
smoke_dry_run = json.loads(raw)
missing_assets = smoke_dry_run.get('missing_assets', {})
scenario_count = int(smoke_dry_run.get('scenario_count', 0))
print('preflight_summary:')
print(json.dumps({
    'scenario_count': scenario_count,
    'missing_assets': missing_assets,
    'config_snapshot': smoke_dry_run.get('config_snapshot', ''),
}, indent=2, sort_keys=True))
if missing_assets:
    raise RuntimeError(f'Resolve missing assets before Step 5: {missing_assets}')
if scenario_count <= 0:
    raise RuntimeError('Preflight found zero scenarios. Verify the Scenario Dreamer environment pack binding before Step 5.')


In [ ]:
# Step 5: Run the multi-scene risk variance study and offline selector probe
import json
import os
import subprocess
import sys


def _run_json_cmd(cmd, label):
    proc = subprocess.run(cmd, text=True, capture_output=True, check=False)
    stdout = proc.stdout.strip()
    stderr = proc.stderr.strip()
    if stdout:
        print(stdout)
    if stderr:
        print(f'[{label}_stderr]')
        print(stderr)
    if proc.returncode != 0:
        diag = {
            'exit_code': proc.returncode,
            'failed_command': ' '.join(cmd),
            'results_root': os.environ.get('SCENARIO_DREAMER_RESULTS_ROOT', ''),
            'recent_stdout': stdout[-4000:],
            'recent_stderr': stderr[-4000:],
        }
        print(f'{label}_diagnostics:')
        print(json.dumps(diag, indent=2, sort_keys=True))
        raise subprocess.CalledProcessError(proc.returncode, cmd)
    return json.loads(stdout)

scenario_indices = os.environ.get('SCENARIO_DREAMER_STUDY_SCENARIO_INDICES', '0,1,2,3,4').strip()
seeds = os.environ.get('SCENARIO_DREAMER_STUDY_SEEDS', '0,1,2,3,4,5,6,7,8,9').strip()
selector_ks = os.environ.get('SCENARIO_DREAMER_STUDY_SELECTOR_KS', '1,2,5,10').strip()
risk_key = os.environ.get('SCENARIO_DREAMER_STUDY_RISK_KEY', 'min_ttc_proxy_s').strip()
visualize = os.environ.get('SCENARIO_DREAMER_STUDY_VISUALIZE', '0').strip().lower() in {'1', 'true', 'yes', 'y', 'on'}
no_lightweight = os.environ.get('SCENARIO_DREAMER_STUDY_NO_LIGHTWEIGHT', '0').strip().lower() in {'1', 'true', 'yes', 'y', 'on'}

cmd = [
    sys.executable,
    'scripts/run_risk_variance_study.py',
    '--scenario-indices', scenario_indices,
    '--seeds', seeds,
    '--selector-k-values', selector_ks,
    '--risk-key', risk_key,
]
if visualize:
    cmd.append('--visualize')
if no_lightweight:
    cmd.append('--no-lightweight')

study_payload = _run_json_cmd(cmd, 'step5_risk_study')
print('risk_study_summary:')
print(json.dumps({
    'run_id': study_payload.get('run_id', ''),
    'run_dir': study_payload.get('run_dir', ''),
    'scene_count': study_payload.get('scene_count', 0),
    'risk_key': study_payload.get('risk_key', ''),
    'aggregate': study_payload.get('aggregate', {}),
    'selector_summary': study_payload.get('selector_summary', {}),
}, indent=2, sort_keys=True))


In [ ]:
# Step 6: Inspect the saved study summary and plot scene-level risk variance
import json
import os
from pathlib import Path

import matplotlib.pyplot as plt

results_root = Path(os.environ['SCENARIO_DREAMER_RESULTS_ROOT'])
run_dirs = sorted([p for p in results_root.iterdir() if p.is_dir() and p.name.endswith('_risk_variance_study')], key=lambda p: p.stat().st_mtime, reverse=True)
print('results_root:', results_root)
print('num_risk_study_run_dirs:', len(run_dirs))
if not run_dirs:
    raise RuntimeError('No risk variance study run directories found. Execute Step 5 first.')

latest = run_dirs[0]
print('latest_run_dir:', latest)
summary_path = latest / 'risk_variance_study_summary.json'
if not summary_path.exists():
    raise RuntimeError(f'Latest run does not contain risk_variance_study_summary.json: {latest}')
summary = json.loads(summary_path.read_text())
print(json.dumps({
    'run_id': summary.get('run_id', ''),
    'risk_key': summary.get('risk_key', ''),
    'aggregate': summary.get('aggregate', {}),
    'selector_summary': summary.get('selector_summary', {}),
}, indent=2, sort_keys=True))

scene_payloads = summary.get('scene_payloads', [])
scene_labels = [scene.get('scenario_name', f"scene_{idx}") for idx, scene in enumerate(scene_payloads)]
risk_stats = [scene.get('risk_variance', {}) for scene in scene_payloads]
means = [item.get('mean') or 0.0 for item in risk_stats]
ranges = [item.get('range') or 0.0 for item in risk_stats]
mins = [item.get('min') or 0.0 for item in risk_stats]
maxs = [item.get('max') or 0.0 for item in risk_stats]

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
axes[0].bar(scene_labels, means)
axes[0].set_title(f"Mean {summary.get('risk_key')} by scene")
axes[0].set_ylabel(summary.get('risk_key'))
axes[0].tick_params(axis='x', rotation=45)

axes[1].bar(scene_labels, ranges)
axes[1].set_title(f"Range of {summary.get('risk_key')} across seeds")
axes[1].set_ylabel('range')
axes[1].tick_params(axis='x', rotation=45)
plt.tight_layout()
plt.show()

for scene, item in zip(scene_payloads, risk_stats):
    print(scene.get('scenario_name'), json.dumps(item, sort_keys=True))


In [ ]:
# Step 7: Inspect the offline selector upper bound and plot improvement vs K
import json
import os
from pathlib import Path

import matplotlib.pyplot as plt

results_root = Path(os.environ['SCENARIO_DREAMER_RESULTS_ROOT'])
run_dirs = sorted([p for p in results_root.iterdir() if p.is_dir() and p.name.endswith('_risk_variance_study')], key=lambda p: p.stat().st_mtime, reverse=True)
latest = run_dirs[0]
summary = json.loads((latest / 'risk_variance_study_summary.json').read_text())
selector_summary = summary.get('selector_summary', {})
scene_payloads = summary.get('scene_payloads', [])

ordered_k = sorted((int(k) for k in selector_summary.keys()))
mean_improvements = [selector_summary[str(k)].get('mean_risk_improvement') or 0.0 for k in ordered_k]
num_improved = [selector_summary[str(k)].get('num_improved') or 0 for k in ordered_k]

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
axes[0].plot(ordered_k, mean_improvements, marker='o')
axes[0].set_title('Offline selector mean risk improvement vs K')
axes[0].set_xlabel('K sampled futures')
axes[0].set_ylabel(f"delta {summary.get('risk_key')}")

axes[1].bar([str(k) for k in ordered_k], num_improved)
axes[1].set_title('Scenes improved by selector upper bound')
axes[1].set_xlabel('K sampled futures')
axes[1].set_ylabel('num scenes improved')
plt.tight_layout()
plt.show()

for scene in scene_payloads:
    compact = {
        'scenario_name': scene.get('scenario_name', ''),
        'selector_probe': scene.get('selector_probe', []),
    }
    print(json.dumps(compact, indent=2, sort_keys=True)[:6000])
